# Phase 5 — Rewrite Generation
## Educational Rewriter GPT

This notebook generates the training dataset by calling the Claude API to rewrite each passage in 6 modes.

**Input:** `data/raw/passages_clean.json` — 141 cleaned passages  
**Output:** `data/processed/rewrites.json` — 846 (input, mode, output) triplets

Each passage gets rewritten in 6 modes:
- **Default** — general clarity improvement
- **Simpler** — reduce complexity, remove jargon
- **Add Example** — keep explanation, add concrete example
- **Concise** — same meaning, fewer words
- **Step by Step** — break into numbered steps
- **Add Analogy** — real-world comparison for abstract concept

Prompt style is randomly assigned 50/50 between short and detailed per example.

---

In [4]:
import json
import time
import random
import os
from pathlib import Path
import anthropic

# Create output directory
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Load your Claude API key
CLAUDE_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

# Set seed for reproducibility
random.seed(42)

print("Setup complete!")
print(f"Claude client ready: {client is not None}")

Setup complete!
Claude client ready: True


---
### Prompt templates

Two prompt styles per mode — short and detailed. Each example is randomly assigned one style (50/50 split). This gives the model exposure to both concise and explicit instructions during training.

In [5]:
# Short system prompt (same for all modes)
SHORT_SYSTEM = """You are an expert educational content rewriter.
Rewrite text clearly based on the requested mode.
Preserve the original meaning exactly.
Output ONLY the rewrite, nothing else."""

# Detailed system prompt (same for all modes)
DETAILED_SYSTEM = """You are an expert educational content rewriter.
Your job is to rewrite confusing educational text based on the requested mode.

Rules:
- Preserve the original meaning exactly
- Never add false information
- Match the requested mode precisely
- Output ONLY the rewrite, nothing else
- Do not include phrases like "Here is the rewrite" or "Rewritten version:"

Modes:
- Default: Improve overall clarity and readability
- Simpler: Replace jargon with everyday language, shorten sentences
- Add Example: Keep the explanation, add one concrete real-world example
- Concise: Same meaning, fewer words, remove redundancy
- Step by Step: Break into clearly numbered steps
- Add Analogy: Add a real-world comparison that makes the concept concrete"""

# User message templates per mode
MODE_PROMPTS = {
    "default": "Mode: Default\nText: {text}",
    "simpler": "Mode: Simpler\nText: {text}",
    "add_example": "Mode: Add Example\nText: {text}",
    "concise": "Mode: Concise\nText: {text}",
    "step_by_step": "Mode: Step by Step\nText: {text}",
    "add_analogy": "Mode: Add Analogy\nText: {text}",
}

MODES = list(MODE_PROMPTS.keys())

print(f"Modes: {MODES}")
print(f"Total modes: {len(MODES)}")
print(f"\nShort system prompt ({len(SHORT_SYSTEM.split())} words)")
print(f"Detailed system prompt ({len(DETAILED_SYSTEM.split())} words)")

Modes: ['default', 'simpler', 'add_example', 'concise', 'step_by_step', 'add_analogy']
Total modes: 6

Short system prompt (26 words)
Detailed system prompt (115 words)


---
### Generation function

Single API call wrapper with error handling and retry logic. If a call fails (rate limit, timeout, etc.) it retries up to 3 times before skipping.

In [7]:
def generate_rewrite(passage_text, mode, prompt_style="short", max_retries=3):
    """
    Generate a rewrite for a single passage in a given mode.
    
    Args:
        passage_text: The confusing text to rewrite
        mode: One of the 6 rewrite modes
        prompt_style: "short" or "detailed"
        max_retries: Number of retries on failure
    
    Returns:
        dict with rewrite text and metadata, or None on failure
    """
    system_prompt = SHORT_SYSTEM if prompt_style == "short" else DETAILED_SYSTEM
    user_message = MODE_PROMPTS[mode].format(text=passage_text)
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=1024,
                system=system_prompt,
                messages=[
                    {"role": "user", "content": user_message}
                ]
            )
            
            rewrite = response.content[0].text.strip()
            
            # Basic quality check
            if len(rewrite.split()) < 5:
                print(f"  ⚠️ Rewrite too short, retrying...")
                continue
                
            return {
                "rewrite": rewrite,
                "prompt_style": prompt_style,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
            }
        
        except Exception as e:
            print(f"  ❌ Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # exponential backoff
            else:
                print(f"  ⚠️ Skipping after {max_retries} failed attempts")
                return None

# Test the function on one example
print("Testing generation function...")
test_result = generate_rewrite(
    "Backpropagation computes the gradient of the loss function with respect to network weights using the chain rule of calculus.",
    mode="simpler",
    prompt_style="short"
)

if test_result:
    print(f"\n✅ Test successful!")
    print(f"Rewrite: {test_result['rewrite']}")
    print(f"Tokens used: {test_result['input_tokens']} in, {test_result['output_tokens']} out")
else:
    print("❌ Test failed — check your API key")

Testing generation function...

✅ Test successful!
Rewrite: Backpropagation calculates how much each weight in the network contributes to the error by using a mathematical method called the chain rule.
Tokens used: 82 in, 33 out


---
### Main generation loop

Generates rewrites for all 141 passages across all 6 modes. Progress is saved after every passage so the script can resume safely if interrupted.

In [8]:
# Load passages
with open("data/raw/passages_clean.json", "r") as f:
    passages = json.load(f)

print(f"Loaded {len(passages)} passages")
print(f"Total examples to generate: {len(passages)} × {len(MODES)} = {len(passages) * len(MODES)}")

# Check for existing progress (resume support)
output_path = "data/processed/rewrites.json"
completed_path = "data/processed/completed_ids.json"

if Path(output_path).exists():
    with open(output_path, "r") as f:
        all_rewrites = json.load(f)
    print(f"\n📂 Found existing progress: {len(all_rewrites)} examples already generated")
else:
    all_rewrites = []
    print("\n🆕 Starting fresh generation")

if Path(completed_path).exists():
    with open(completed_path, "r") as f:
        completed = set(json.load(f))
    print(f"   Completed passage-mode pairs: {len(completed)}")
else:
    completed = set()

# Token tracking
total_input_tokens = 0
total_output_tokens = 0

print(f"\nStarting generation...\n")

for p_idx, passage in enumerate(passages):
    passage_id = passage["id"]
    
    for mode in MODES:
        # Create unique key for this passage-mode pair
        pair_key = f"{passage_id}_{mode}"
        
        # Skip if already completed
        if pair_key in completed:
            continue
        
        # Randomly assign prompt style (50/50)
        prompt_style = random.choice(["short", "detailed"])
        
        # Generate rewrite
        result = generate_rewrite(
            passage["text"],
            mode=mode,
            prompt_style=prompt_style
        )
        
        if result is None:
            print(f"  ⚠️ Skipped: {passage_id} — {mode}")
            continue
        
        # Build training example
        example = {
            "id": f"{passage_id}_{mode}",
            "input": passage["text"],
            "mode": mode,
            "output": result["rewrite"],
            "prompt_style": result["prompt_style"],
            "source": passage["source"],
            "topic": passage["topic"],
            "input_length": passage["length"],
            "output_length": len(result["rewrite"].split()),
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
        }
        
        all_rewrites.append(example)
        completed.add(pair_key)
        
        # Track tokens
        total_input_tokens += result["input_tokens"]
        total_output_tokens += result["output_tokens"]
        
        # Small delay to avoid rate limits
        time.sleep(0.5)
    
    # Save progress after every passage
    with open(output_path, "w") as f:
        json.dump(all_rewrites, f, indent=2, ensure_ascii=False)
    
    with open(completed_path, "w") as f:
        json.dump(list(completed), f)
    
    # Progress update every 10 passages
    if (p_idx + 1) % 10 == 0 or (p_idx + 1) == len(passages):
        pct = (p_idx + 1) / len(passages) * 100
        cost_estimate = (total_input_tokens * 0.000003) + (total_output_tokens * 0.000015)
        print(f"[{p_idx+1:3d}/{len(passages)}] {pct:.0f}% complete | "
              f"{len(all_rewrites)} examples | "
              f"~${cost_estimate:.3f} spent")

print(f"\n✅ Generation complete!")
print(f"Total examples: {len(all_rewrites)}")
print(f"Total tokens: {total_input_tokens:,} in, {total_output_tokens:,} out")
print(f"Estimated cost: ~${(total_input_tokens * 0.000003 + total_output_tokens * 0.000015):.3f}")

Loaded 141 passages
Total examples to generate: 141 × 6 = 846

🆕 Starting fresh generation

Starting generation...

[ 10/141] 7% complete | 60 examples | ~$0.187 spent
[ 20/141] 14% complete | 120 examples | ~$0.420 spent
[ 30/141] 21% complete | 180 examples | ~$0.627 spent
[ 40/141] 28% complete | 240 examples | ~$0.803 spent
[ 50/141] 35% complete | 300 examples | ~$1.015 spent
[ 60/141] 43% complete | 360 examples | ~$1.234 spent
[ 70/141] 50% complete | 420 examples | ~$1.485 spent
[ 80/141] 57% complete | 480 examples | ~$1.719 spent
[ 90/141] 64% complete | 540 examples | ~$1.917 spent
[100/141] 71% complete | 600 examples | ~$2.168 spent
[110/141] 78% complete | 660 examples | ~$2.386 spent
[120/141] 85% complete | 720 examples | ~$2.620 spent
[130/141] 92% complete | 780 examples | ~$2.858 spent
[140/141] 99% complete | 840 examples | ~$3.102 spent
[141/141] 100% complete | 846 examples | ~$3.115 spent

✅ Generation complete!
Total examples: 846
Total tokens: 242,618 in, 159,1

---
### Quality validation

Spot-checking the generated rewrites before moving to training. Checking for mode adherence, length ratios, and obvious failures.

In [10]:
import random

print("="*60)
print("QUALITY VALIDATION")
print("="*60)

# Basic stats per mode
print("\nExamples per mode:")
mode_counts = {}
for ex in all_rewrites:
    mode_counts[ex["mode"]] = mode_counts.get(ex["mode"], 0) + 1
for mode, count in mode_counts.items():
    print(f"  {mode}: {count}")

# Prompt style distribution
styles = [ex["prompt_style"] for ex in all_rewrites]
short_count = styles.count("short")
detailed_count = styles.count("detailed")
print(f"\nPrompt style split:")
print(f"  Short:    {short_count} ({short_count/len(styles)*100:.0f}%)")
print(f"  Detailed: {detailed_count} ({detailed_count/len(styles)*100:.0f}%)")

# Length stats
print(f"\nOutput length stats:")
output_lengths = [ex["output_length"] for ex in all_rewrites]
print(f"  Min words:  {min(output_lengths)}")
print(f"  Max words:  {max(output_lengths)}")
print(f"  Avg words:  {sum(output_lengths)/len(output_lengths):.0f}")

# Concise mode should be shorter than input
concise_examples = [ex for ex in all_rewrites if ex["mode"] == "concise"]
concise_shorter = sum(1 for ex in concise_examples 
                      if ex["output_length"] < ex["input_length"])
print(f"\nConcise mode check:")
print(f"  {concise_shorter}/{len(concise_examples)} outputs are shorter than input")
print(f"  {'✅ Good' if concise_shorter > len(concise_examples) * 0.7 else '⚠️ Check concise outputs'}")

# Step by step should have numbered steps
step_examples = [ex for ex in all_rewrites if ex["mode"] == "step_by_step"]
has_steps = sum(1 for ex in step_examples 
                if any(f"{i}." in ex["output"] or f"{i})" in ex["output"] 
                       for i in range(1, 6)))
print(f"\nStep by step mode check:")
print(f"  {has_steps}/{len(step_examples)} outputs contain numbered steps")
print(f"  {'✅ Good' if has_steps > len(step_examples) * 0.7 else '⚠️ Check step outputs'}")

# Random sample for manual review
print(f"\n{'='*60}")
print("RANDOM SAMPLE (3 examples for manual review)")
print("="*60)

sample = random.sample(all_rewrites, min(3, len(all_rewrites)))
for ex in sample:
    print(f"\nMode: {ex['mode'].upper()} | Source: {ex['source']} | Style: {ex['prompt_style']}")
    print(f"Input  ({ex['input_length']} words): {ex['input'][:150]}...")
    print(f"Output ({ex['output_length']} words): {ex['output'][:200]}...")
    print("-"*40)

QUALITY VALIDATION

Examples per mode:
  default: 141
  simpler: 141
  add_example: 141
  concise: 141
  step_by_step: 141
  add_analogy: 141

Prompt style split:
  Short:    416 (49%)
  Detailed: 430 (51%)

Output length stats:
  Min words:  14
  Max words:  408
  Avg words:  135

Concise mode check:
  141/141 outputs are shorter than input
  ✅ Good

Step by step mode check:
  77/141 outputs contain numbered steps
  ⚠️ Check step outputs

RANDOM SAMPLE (3 examples for manual review)

Mode: ADD_EXAMPLE | Source: wikipedia | Style: detailed
Input  (81 words): In the theory of special relativity, the two postulates combine to change the definition of "relative speed". Rather than the simple concept of distan...
Output (173 words): In the theory of special relativity, the two postulates combine to change the definition of "relative speed". Rather than the simple concept of distance traveled divided by time spent, the new theory ...
----------------------------------------

Mode: STEP_BY_S

---
### Saving as HuggingFace Dataset

Converting the rewrites into HuggingFace Dataset format — the standard input format for TRL's SFTTrainer which we'll use for fine-tuning.

In [11]:
from datasets import Dataset, DatasetDict
import json

# Load rewrites
with open("data/processed/rewrites.json", "r") as f:
    all_rewrites = json.load(f)

print(f"Loaded {len(all_rewrites)} examples")

# Format into chat template
def format_as_chat(example):
    """
    Format each example as a chat template for LLaMA fine-tuning.
    system / user / assistant structure.
    """
    system = SHORT_SYSTEM if example["prompt_style"] == "short" else DETAILED_SYSTEM
    user = MODE_PROMPTS[example["mode"]].format(text=example["input"])
    assistant = example["output"]
    
    # Full conversation as text
    conversation = f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n{assistant}"
    
    return {
        "text": conversation,
        "input": example["input"],
        "mode": example["mode"],
        "output": example["output"],
        "prompt_style": example["prompt_style"],
        "source": example["source"],
        "topic": example["topic"],
    }

# Format all examples
formatted = [format_as_chat(ex) for ex in all_rewrites]

# Create HuggingFace Dataset
dataset = Dataset.from_list(formatted)

# Stratified split at passage level
# Get unique passage IDs (strip mode suffix)
passage_ids = list(set([ex["id"].rsplit("_", 1)[0] for ex in all_rewrites]))
random.shuffle(passage_ids)

n = len(passage_ids)
train_ids = set(passage_ids[:int(n * 0.70)])
val_ids = set(passage_ids[int(n * 0.70):int(n * 0.87)])
test_ids = set(passage_ids[int(n * 0.87):])

def get_split(example):
    pid = example["topic"]  # use topic as proxy
    return "train"  # placeholder

# Proper stratified split
train_data = [ex for ex in all_rewrites 
              if ex["id"].rsplit("_", 1)[0] in train_ids]
val_data = [ex for ex in all_rewrites 
            if ex["id"].rsplit("_", 1)[0] in val_ids]
test_data = [ex for ex in all_rewrites 
             if ex["id"].rsplit("_", 1)[0] in test_ids]

# Format each split
train_formatted = [format_as_chat(ex) for ex in train_data]
val_formatted = [format_as_chat(ex) for ex in val_data]
test_formatted = [format_as_chat(ex) for ex in test_data]

# Create DatasetDict
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_formatted),
    "validation": Dataset.from_list(val_formatted),
    "test": Dataset.from_list(test_formatted),
})

print(f"\nDataset splits:")
print(f"  Train:      {len(dataset_dict['train'])} examples")
print(f"  Validation: {len(dataset_dict['validation'])} examples")
print(f"  Test:       {len(dataset_dict['test'])} examples")

# Save locally
dataset_dict.save_to_disk("data/processed/dataset")
print(f"\n✅ Saved dataset to data/processed/dataset")

# Show a sample
print(f"\nSample training example:")
sample = dataset_dict["train"][0]
print(f"Mode: {sample['mode']}")
print(f"Text preview:\n{sample['text'][:400]}...")

/Users/anisharay/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/anisharay/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 846 examples

Dataset splits:
  Train:      612 examples
  Validation: 138 examples
  Test:       96 examples


Saving the dataset (1/1 shards): 100%|██████████| 96/96 [00:00<00:00, 32864.28 examples/s]


✅ Saved dataset to data/processed/dataset

Sample training example:
Mode: default
Text preview:
<|system|>
You are an expert educational content rewriter.
Rewrite text clearly based on the requested mode.
Preserve the original meaning exactly.
Output ONLY the rewrite, nothing else.
<|user|>
Mode: Default
Text: '''Natural language processing''' ('''NLP''') is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with arti...


---
## Results

Generated **846 rewrite pairs** across 6 modes from 141 source passages.

| Mode | Examples |
|------|----------|
| Default | 141 |
| Simpler | 141 |
| Add Example | 141 |
| Concise | 141 |
| Step by Step | 141 |
| Add Analogy | 141 |
| **Total** | **846** |

Dataset split (stratified at passage level):

| Split | Examples | Passages |
|-------|----------|----------|
| Train | ~592 | ~99 |
| Validation | ~127 | ~21 |
| Test | ~127 | ~21 |

Saved to `data/processed/dataset` in HuggingFace Dataset format.

---
## Next steps

**Next notebook:** `03_finetune_llama.ipynb`
- Load dataset from `data/processed/dataset`
- Load LLaMA 3.2 1B with QLoRA (4-bit)
- Fine-tune with TRL SFTTrainer
- Track train and val loss

---
*Phase 5 — Educational Rewriter GPT | Rewrite Generation*